In [2]:
!pip install tensorflow

   ---------------------------------------- 0.0/375.9 MB ? eta -:--:--
   ---------------------------------------- 0.0/375.9 MB ? eta -:--:--
   ---------------------------------------- 0.5/375.9 MB 1.9 MB/s eta 0:03:21
   ---------------------------------------- 1.6/375.9 MB 3.4 MB/s eta 0:01:52
   ---------------------------------------- 1.8/375.9 MB 3.7 MB/s eta 0:01:41
   ---------------------------------------- 2.1/375.9 MB 2.3 MB/s eta 0:02:43
   ---------------------------------------- 2.4/375.9 MB 2.3 MB/s eta 0:02:45
   ---------------------------------------- 2.6/375.9 MB 2.1 MB/s eta 0:02:56
   ---------------------------------------- 2.6/375.9 MB 2.1 MB/s eta 0:02:56
   ---------------------------------------- 2.9/375.9 MB 1.7 MB/s eta 0:03:45
   ---------------------------------------- 3.1/375.9 MB 1.7 MB/s eta 0:03:45
   ---------------------------------------- 3.4/375.9 MB 1.5 MB/s eta 0:04:03
   ---------------------------------------- 3.7/375.9 MB 1.5 MB/s eta 0:04:03


ERROR: Could not install packages due to an OSError: [WinError 2] The system cannot find the file specified: 'C:\\Python311\\Scripts\\wheel.exe' -> 'C:\\Python311\\Scripts\\wheel.exe.deleteme'



In [5]:
import os
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from flask import Flask, request, render_template
from tensorflow.keras.preprocessing.image import load_img, img_to_array, ImageDataGenerator
from sklearn.metrics import classification_report
from sklearn.utils.class_weight import compute_class_weight

ModuleNotFoundError: No module named 'tensorflow'

In [ ]:
# Define dataset path
dataset_path = "rgc_data"

In [ ]:
# Mapping Retinal OCT labels to Alzheimer's categories
alzheimers_labels = {
    "NORMAL": "CN",  # Cognitively Normal
    "DRUSEN": "MCI",  # Mild Cognitive Impairment
    "CNV": "AD",  # Alzheimer's Disease
    "DME": "AD"  # Alzheimer's Disease
}

# Collect image paths and labels
train_image_paths, train_labels = [], []
test_image_paths, test_labels = [], []
val_image_paths, val_labels = [], []

for split, image_list, label_list in [
    ("train", train_image_paths, train_labels), 
    ("test", test_image_paths, test_labels), 
    ("val", val_image_paths, val_labels)
]:
    for category in alzheimers_labels.keys():
        folder_path = os.path.join(dataset_path, split, category)
        if not os.path.exists(folder_path):
            print(f"Warning: {folder_path} does not exist. Skipping...")
            continue
        for img_name in os.listdir(folder_path):
            img_path = os.path.join(folder_path, img_name)
            image_list.append(img_path)
            label_list.append(alzheimers_labels[category])

# Load images with augmentation
datagen = ImageDataGenerator(
    rotation_range=20,
    width_shift_range=0.1,
    height_shift_range=0.1,
    shear_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True,
    fill_mode="nearest"
)

In [ ]:
def load_images(image_paths, labels, img_size=(300, 300)):
    X, y = [], []
    for img_path, label in zip(image_paths, labels):
        try:
            img = load_img(img_path, target_size=img_size)
            img_array = img_to_array(img) / 255.0  # Normalize
            X.append(img_array)
            y.append(label)
        except Exception as e:
            print(f"Error loading image {img_path}: {e}")
    return np.array(X), np.array(y)

# Load datasets
train_X, train_y = load_images(train_image_paths, train_labels)
test_X, test_y = load_images(test_image_paths, test_labels)
val_X, val_y = load_images(val_image_paths, val_labels)

# Encode labels
label_map = {"CN": 0, "MCI": 1, "AD": 2}
train_y = np.array([label_map[label] for label in train_y])
test_y = np.array([label_map[label] for label in test_y])
val_y = np.array([label_map[label] for label in val_y])

# Compute class weights
class_weights = compute_class_weight(class_weight='balanced', classes=np.unique(train_y), y=train_y)
class_weights = dict(enumerate(class_weights))

In [ ]:
# Build improved model
def build_model(input_shape=(300, 300, 3), num_classes=3):
    base_model = keras.applications.EfficientNetB3(weights='imagenet', include_top=False, input_shape=input_shape)
    for layer in base_model.layers[-50:]:  # Fine-tune last 50 layers
        layer.trainable = True
    
    model = keras.Sequential([
        base_model,
        layers.GlobalAveragePooling2D(),
        layers.Dense(256, activation='relu'),
        layers.Dropout(0.4),
        layers.Dense(128, activation='relu'),
        layers.Dropout(0.3),
        layers.Dense(num_classes, activation='softmax')
    ])
    model.compile(optimizer=keras.optimizers.Adam(learning_rate=1e-5), 
                  loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    return model


In [ ]:
# Train model
model = build_model()
lr_scheduler = keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5, verbose=1)
history = model.fit(datagen.flow(train_X, train_y, batch_size=32), 
                    validation_data=(val_X, val_y), 
                    epochs=10, 
                    class_weight=class_weights, 
                    callbacks=[lr_scheduler])

# Save trained model
model.save('alzheimers_rgc_model.h5')
print("Model trained and saved successfully!")

# Predict on test data
test_predictions = model.predict(test_X)
test_pred_labels = np.argmax(test_predictions, axis=1)

# Evaluate model on validation set
val_predictions = model.predict(val_X)
val_pred_labels = np.argmax(val_predictions, axis=1)

print("Classification Report (Validation Set):")
print(classification_report(val_y, val_pred_labels, target_names=["CN", "MCI", "AD"]))